In [1]:
#IMPORT MODULES 
import pandas as pd
import pickle
import matplotlib.pyplot as plot
from datetime import datetime, date
import time
import numpy as np
import copy
import math
#use parseData.py to parse the data from the raw json files.  Use sensor.py to generate data (you'll need an mqtt host) and the sensorUI to save off the data.
#print(myDf)
pd.set_option("display.max.columns", None)


In [2]:
#READ IN DATA
#test dataframe setup
#testpd=pd.DataFrame(index=np.arange(20),columns=['id','temp_p','temp_h','pressure','humidity','date','time'])
#testpd.loc[1]={'1','1','1','1','1','1','1'}
myDf=pd.read_pickle('pkl/sensorData_JanFebMarAprMay_byPi.pkl')#This pkl file is soreted by RPi.  The Raw data (JSON Files) use a custom configuration.  You'll need to modify the parseData.py to use the custom configuration
myDf=myDf.sort_values(by='time').reset_index(drop=True)

In [ ]:
#THIS IS AN OPTIONAL CELL!  THIS SPLITS UP THE DATAFRAME INTO EQUAL-SIZED PARTS.  USE THIS IF IT TAKES TOO LONG TO RUN
print(myDf.size)
S = 3
N = int(len(myDf)/S)
frames = [ myDf.iloc[i*S:(i+1)*S].copy() for i in range(N+1) ]

In [3]:
#THIS CELL ADDS DUPLICATE POINTS IN THE GAPS BETWEEN MEASUREMENTS; 
#This is necessary to measure the gaps between measurements are included as measurable points for the NN

old_tick=None
diff=None
count=0
t=1 #arbitrary value.  I assume a single missed reading is anomalous.
elements=[]
elementIds=[]
for i in range(0,len(myDf)):
    element=dict(myDf.loc[i])
    elemId=element['id']
    if element['id'] not in elementIds:
        elements.append(element)
        elementIds.append(element['id'])
        #print('1:'+element['id'])
    else:
        ans=[elem for elem in elements if elem['id'] == element['id']]
        diff=element['time']-ans[0]['time']
        #since I found a gap, I need to remove the old element, and replace it with the later one
        elements.remove(ans[0])
        #elementIds.remove(elemId)
        elements.append(element)
        #elementIds.append(elemId)
        if diff>=30+t:
            #print('at index: '+ str(i))
            #print(element['id'])
            #print(element['time'])
            #print(ans[0]['id'])
            #print(ans[0]['time'])
            #print(diff)
            numInserts=int(diff/30)-1
            #print(numInserts)#tells me how mny new additions per element.
            #print(str(numInserts)+' ID:'+element['id'])
            #print('previous time:'+str(element['time']))
            temp_df=pd.DataFrame(index=np.arange(numInserts),columns=['id','temp_p','temp_h','pressure','humidity','date','time','Y'])
            for ins in range(numInserts):
                newDict=copy.deepcopy(ans[0])
                newDict['time']=newDict['time']+(30*(ins+1))
                #gotta ask whether or not I should include NaN or identical preceeding number
                newDict['temp_p']=np.NaN
                newDict['temp_h']=np.NaN
                newDict['pressure']=np.NaN
                newDict['humidity']=np.NaN
                newDict['Y']=-1#since I'm adding data, I need to label it as anomalous(static/missing data anomaly)
                #print(newDict['time'])
                #print(element['time'])
                #pd.DataFrame(newDict,index=[0])
                temp_df.loc[ins]=newDict
            myDf=myDf.append(temp_df,ignore_index=True)
myDf=myDf.sort_values(by='time').reset_index(drop=True)
myDf.to_pickle('sensors_2022JanToAprExpandeded.pkl')#this is just in case the computer loses power.
#                print(newDict)

In [45]:
#THIS IS AN OPTIONAL CELL.  IF FOR SOME REASON THE ABOVE CELL DOESN'T SET THE LABEL, USE THIS CELL
#myDf['Y']=1
#labeling for NAN data
noNanData=False #adding space and anomaly labels between the original data so that the sample rate is continuous across the whole set.  All data is labeled NaN
noDropoutLabels=True #adding anomaly labels if data suddenly goes to 0 for temp_p data.
noStaticLabels=False #adding anomaly labels if data reports the exact same number between samples
prevGoodinfo=[{'id':1,'temp_p':np.nan,'temp_h':np.nan,'pressure':np.nan,'humidity':np.nan},{'id':2,'temp_p':np.nan,'temp_h':np.nan,'pressure':np.nan,'humidity':np.nan},{'id':3,'temp_p':np.nan,'temp_h':np.nan,'pressure':np.nan,'humidity':np.nan},{'id':4,'temp_p':np.nan,'temp_h':np.nan,'pressure':np.nan,'humidity':np.nan},{'id':5,'temp_p':np.nan,'temp_h':np.nan,'pressure':np.nan,'humidity':np.nan},{'id':6,'temp_p':np.nan,'temp_h':np.nan,'pressure':np.nan,'humidity':np.nan},{'id':7,'temp_p':np.nan,'temp_h':np.nan,'pressure':np.nan,'humidity':np.nan},{'id':8,'temp_p':np.nan,'temp_h':np.nan,'pressure':np.nan,'humidity':np.nan},{'id':9,'temp_p':np.nan,'temp_h':np.nan,'pressure':np.nan,'humidity':np.nan},{'id':10,'temp_p':np.nan,'temp_h':np.nan,'pressure':np.nan,'humidity':np.nan}]
#prevGoodTemp_h=[{'id':1,'temp_h':np.nan,},{'id':2,'temp_h':np.nan},{'id':3,'temp_h':np.nan},{'id':4,'temp_h':np.nan},{'id':5,'temp_h':np.nan},{'id':6,'temp_h':np.nan},{'id':7,'temp_h':np.nan},{'id':8,'temp_h':np.nan},{'id':9,'temp_h':np.nan},{'id':10,'temp_h':np.nan}]
#prevGoodPressure=[{'id':1,'pressure':np.nan,},{'id':2,'pressure':np.nan},{'id':3,'pressure':np.nan},{'id':4,'pressure':np.nan},{'id':5,'pressure':np.nan},{'id':6,'pressure':np.nan},{'id':7,'pressure':np.nan},{'id':8,'pressure':np.nan},{'id':9,'pressure':np.nan},{'id':10,'pressure':np.nan}]
#prevGoodHumidity=[{'id':1,'humidity':np.nan,},{'id':2,'humidity':np.nan},{'id':3,'humidity':np.nan},{'id':4,'humidity':np.nan},{'id':5,'humidity':np.nan},{'id':6,'humidity':np.nan},{'id':7,'humidity':np.nan},{'id':8,'humidity':np.nan},{'id':9,'humidity':np.nan},{'id':10,'humidity':np.nan}]
for j in range(len(myDf)):
    currId=myDf.loc[j]['id']
    currTemp_p=myDf.loc[j]['temp_p']
    currTemp_h=myDf.loc[j]['temp_h']
    curr_pressure=myDf.loc[j]['pressure']
    curr_humidity=myDf.loc[j]['humidity']
    ans=[elem for elem in prevGoodinfo if elem['id'] == currId][0]#gets the proper variable for the sensor
    
    if noNanData:
        if math.isnan(myDf.loc[j]['temp_p']) and math.isnan(myDf.loc[j]['temp_h']) and math.isnan(myDf.loc[j]['pressure']) and math.isnan(myDf.loc[j]['humidity']):
            myDf.loc[j,'Y']=-1
        else:
            myDf.loc[j,'Y']=1

    if noDropoutLabels:
        if ans['temp_p'] is not np.nan: #there's previously set data
            if (currTemp_p==0) and abs(prevGoodinfo[currId-1]['temp_p']-currTemp_p)>= .25: #there was a significant jump from reading to reading, indicating that this 0 value is a startup error
                #myDf.loc[j,'Y']=-1
                print('id: ' + str(currId) + 'time: '+ str(myDf.loc[j]['time']) +' prevGoodTemp_p: '+ str(prevGoodinfo[currId-1]['temp_p']) + ', curr temp: '+ str(currTemp_p))
            else:
                prevGoodinfo[currId-1]['temp_p']=currTemp_p#set the last good temp_p.  I use -1 because index starts at 0
        else: #need to set good previous temp_p data
            prevGoodinfo[currId-1]['temp_p']=currTemp_p#set the last good temp_p.  I use -1 because index starts at 0

    if noStaticLabels:
        if prevGoodHumidity is not np.nan:
            if prevGoodHumidity==myDf.loc[j]['humidity']:#the readings are identical
                print('id: ' + str(myDf.loc[j]['id']) + 'time: '+ str(myDf.loc[j]['time'])  + ' prevGoodHumidity: '+ str(prevGoodHumidity) + ', curr Humidity: '+ str(myDf.loc[j]['humidity']))
                #myDf.loc[j,'Y']=-1
            else:
                prevGoodHumidity=myDf.loc[j]['humidity']    
        else:
            prevGoodHumidity=myDf.loc[j]['humidity']


id: 7time: 1643132347.337406 prevGoodTemp_p: 20.160417556762695, curr temp: 0.0
id: 7time: 1643132500.616838 prevGoodTemp_p: 20.15416717529297, curr temp: 0.0
id: 3time: 1643633366.4567823 prevGoodTemp_p: 10.004165649414062, curr temp: 0.0
id: 3time: 1643633396.5348415 prevGoodTemp_p: 10.012500762939453, curr temp: 0.0


IndexError: list index out of range

In [14]:
# This cell adds additional chronological data and places the data into dictionaries more easily suited for plotting
# This isn't done with the other for loop because there would be elements missed due to the increasing size of the list
leapYearCheck=2020#2020 was the last leap year.  take the current year, subtract 2020, and if the number evenly divides by 4, its a leap year (indicating that feburary has 29 days)
monthMult={'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12}
DayCheck={1:31,2:28,3:31,4:30,5:31,6:30,7:31,8:31,9:30,10:31,11:30,12:31}
dayOfYear=[]
timeOfDay=0
temp_p={'1':[],'2':[],'3':[],'4':[],'5':[],'6':[],'7':[],'8':[],'9':[],'10':[],'11':[],'12':[],'13':[],'14':[],'15':[],'16':[],'17':[]}#this cell's dictionaries is sized to deal with IDs from RAW data.  I modified trhe pkl to be IDs of specific RPis.  It works for both, but IDs 11-17 will be empty if you use the pre-made pkl data or run the cell that sorts everything by RPi
temp_h={'1':[],'2':[],'3':[],'4':[],'5':[],'6':[],'7':[],'8':[],'9':[],'10':[],'11':[],'12':[],'13':[],'14':[],'15':[],'16':[],'17':[]}
pressure={'1':[],'2':[],'3':[],'4':[],'5':[],'6':[],'7':[],'8':[],'9':[],'10':[],'11':[],'12':[],'13':[],'14':[],'15':[],'16':[],'17':[]}
humidity={'1':[],'2':[],'3':[],'4':[],'5':[],'6':[],'7':[],'8':[],'9':[],'10':[],'11':[],'12':[],'13':[],'14':[],'15':[],'16':[],'17':[]}
times={'1':[],'2':[],'3':[],'4':[],'5':[],'6':[],'7':[],'8':[],'9':[],'10':[],'11':[],'12':[],'13':[],'14':[],'15':[],'16':[],'17':[]}
agg_time=[]

for i in range(0,len(myDf)):
    datStr=myDf.loc[i]['date']
    year=datStr[0:4]
    mon=datStr[4:7]
    day=datStr[7:9]
    tim=myDf.loc[i]['time']
    dt=datetime.fromtimestamp(tim)
    timeOfDay=dt.hour*3600 + dt.minute*60 +dt.second
    #index23424 changes from Jan 25 to Jan 26
    #print(year+' '+mon+' '+day)
    #print((int(year)-leapYearCheck)%4)
    if mon=='Feb' and int(year)-leapYearCheck%4==0:#we're in a leap year.  Only do this for data one year at a tiem, else we need to regularly overwrite Feb
        DayCheck[2]=29
    #check the month, use the value of the monthMult for that month to know how many other months you need to add the days from.
    #for example, if the month is Jan, you don;t use any other months since the value is 1.  If the month is Feb, you use the days from the date plus the DayCheck value for Jan
    currentMonth=monthMult[mon]
    daysToAdd=0
    for j in range(1,currentMonth+1):
        if not (j==currentMonth):
            oldMonthDays=DayCheck[j]
            daysToAdd=daysToAdd+oldMonthDays
    daysToAdd=daysToAdd+int(day)
    #print(daysToAdd)
    #dayOfYear.append(daysToAdd)
    myDf.at[i,'doy']=daysToAdd
    myDf.at[i,'tod']=timeOfDay
    
    #splitting data for plotting
    temp_p[myDf.loc[i]['id']].append(myDf.loc[i]['temp_p'])#a sudden drop to 0C indicates an anomaly (a re-initialization of the sensor)
    temp_h[myDf.loc[i]['id']].append(myDf.loc[i]['temp_h'])
    pressure[myDf.loc[i]['id']].append(myDf.loc[i]['pressure'])
    humidity[myDf.loc[i]['id']].append(myDf.loc[i]['humidity'])
    times[myDf.loc[i]['id']].append(myDf.loc[i]['time'])#if there's a gap of longer than 30+t where t is an offset (in seconds) for any measurement, flag the gap as an anomaly (lost connection/restart)
    #Note:  With the above addition of space between points, gaps in data are seen as NaN across all columns and a label (Y) of -1
myDf.to_pickle('pkl/sensors_2022JanToAprProcessed.pkl')
dataPlotDump={'temp_p':temp_p,'temp_h':temp_h,'pressure':pressure,'humidity':humidity,'times':times}
# save dictionary to pickle file
with open('pkl/dataPlotDump2022JanToApr.pkl', 'wb') as file:
    pickle.dump(dataPlotDump, file, protocol=pickle.HIGHEST_PROTOCOL)

In [2]:
myDf=pd.read_pickle('pkl/dataPlotDump2022JanToApr.pkl')
pd.set_option("display.max.columns", None)

In [3]:
ans=myDf['temp_p']

In [ ]:
#PLOTTING CELL

def plotFig(id:str,data, col:str):
    fig,ax=plot.subplots(figsize=(100,8))
    plot.plot(times[id],data,'.',color=col)
    rng=np.arange(min(times[id]),max(times[id]),50000)
    ax.set_xticks(rng)
    return fig,ax
#myfig1, myax1=plotFig('1',humidity['1'],'r')
#myfig2, myax2=plotFig('2',humidity['2'],'m')
#myfig3, myax3=plotFig('3',humidity['3'],'c')
#myfig4, myax4=plotFig('4',humidity['4'],'y')
#myfig5, myax5=plotFig('5',humidity['5'],'k')
#myfig6, myax6=plotFig('6',humidity['6'],'g')
#myfig7, myax7=plotFig('7',humidity['7'],'tab:red')
#myfig8, myax8=plotFig('8',humidity['8'],'tab:orange')
#myfig9, myax9=plotFig('9',humidity['9'],'tab:green')
#myfig10, myax10=plotFig('10',humidity['10'],'tab:blue')
plot.figure(figsize=(200,8))
fig,ax=plot.subplots(figsize=(100,8))
plot.plot(times['9'],temp_p['9'],'r.')
plot.plot(times['9'],temp_h['9'],'m.')
#plot.plot(times['9'],pressure['9'],'c.')
plot.plot(times['9'],humidity['9'],'y.')
#plot.plot(times['5'],temp_p['5'],'k.')
#plot.plot(times['6'],temp_p['6'],'.',color='tab:blue')
#plot.plot(times['7'],temp_p['7'],'.',color='tab:orange')
#plot.plot(times['8'],temp_p['8'],'.',color='tab:green')
#plot.plot(times['9'],temp_p['9'],'g.')
#plot.plot(times['10'],temp_p['10'],'.',color='tab:red')
##plot.xlim([min(time['1'])-1,max(time['1'])+1])
##plot.ticklabel_format(style='plain')
rng=np.arange(min(times['9']),max(times['9']),50000)
ax.set_xticks(rng)
##plot.plot(time['1'],temp_h['1'],'y-')
##plot.plot(time['1'],pressure['1'],'y-')
##plot.plot(time['1'],humidity['1'],'g-')